加入multi-query: 由于用户输入的原始问题往往存在模糊性或关键词缺失，通过 LLM 等手段 将一个问题改写成多个视角不同的等价问题，再进行并发检索，最后合并结果


加入rerank: 搜索系统有两部分 一是召回 通过模型从数据库大量的数据中找到相近的10~50条内容, 计算代价小,但是由于这个召回的过程是embedding的匹配的过程,所以对于细节部分可能处理不好,是粗粒度的搜索; 二是重排(rerank), 采用交叉编码器架构, 全交互注意力, 计算代价大, 负责将召回的数据进行排序,将匹配度高的返回给用户.


为什么计算代价有区别: 向量匹配只需要算两个向量的乘积得出相似度就可以了, 因为是单纯的向量相乘所以没有语义能力, 重排是计算注意力,计算量大

In [ ]:
import torch
print(torch.__version__)

In [ ]:
import os
import torch
import faiss
import pickle
import librosa
import numpy as np
from tqdm import tqdm
from pathlib import Path
from transformers import AutoModel, AutoTokenizer

# ==========================================
# 1. 向量数据库封装 (Faiss)
# ==========================================
class FaissIndex:
    def __init__(self, dim):
        # IndexFlatIP 用于内积计算，配合 L2 归一化即为余弦相似度
        self.index = faiss.IndexFlatIP(dim)
        self.metadata = []

    def add(self, vectors, metas):
        # 显式转换确保类型正确
        vectors = np.ascontiguousarray(vectors).astype("float32")
        self.index.add(vectors)
        self.metadata.extend(metas)

    def save(self, path):
        faiss.write_index(self.index, f"{path}.index")
        with open(f"{path}.meta", "wb") as f:
            pickle.dump(self.metadata, f)

    def load(self, path):
        self.index = faiss.read_index(f"{path}.index")
        with open(f"{path}.meta", "rb") as f:
            self.metadata = pickle.load(f)

# ==========================================
# 2. CLAP 模型封装
# ==========================================
from transformers import ClapProcessor, ClapModel

class CLAPWrapper:
    def __init__(self, model_path, device="cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        self.model = ClapModel.from_pretrained(model_path).to(self.device)
        self.processor = ClapProcessor.from_pretrained(model_path)

    def get_audio_embedding(self, audio):
        # 使用官方 Processor 处理音频，它会自动处理采样率、对齐和特征提取
        # 注意：sampling_rate 必须与加载音频时一致
        inputs = self.processor(
            audio=[audio],
            return_tensors="pt",
            sampling_rate=48000
        ).to(self.device)
        
        with torch.no_grad():
            # 此时使用 model 直接处理封装后的 inputs
            outputs = self.model.get_audio_features(**inputs)
            emb = outputs.pooler_output.cpu().numpy()
        return emb
    
    def get_text_embedding(self, text):
        inputs = self.processor(
            text=[text],
            return_tensors="pt",
            padding=True
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model.get_text_features(**inputs)
            emb = outputs.pooler_output.cpu().numpy()

        return emb

# ==========================================
# 3. 工具函数
# ==========================================
def load_audio(path, sr=48000, max_duration=10):
    y, _ = librosa.load(path, sr=sr, mono=True)

    max_len = sr * max_duration
    if len(y) > max_len:
        y = y[:max_len]
    elif len(y) < max_len:
        y = np.pad(y, (0, max_len - len(y)))

    return y

# ==========================================
# 4. 主构建逻辑
# ==========================================
# 配置参数
AUDIO_DIR = "data/raw_audio"
MODEL_PATH = "/root/autodl-tmp/clap-model"
SAVE_PATH = "data/embeddings/audio_index"

def build():
    # 环境检查
    if not os.path.exists(MODEL_PATH):
        print(f"错误: 找不到模型路径 {MODEL_PATH}")
        return

    # 初始化模型
    model = CLAPWrapper(MODEL_PATH)
    vectors = []
    metas = []

    # 递归查找所有 .wav 文件，排除 .json 或其他杂质
    audio_path_obj = Path(AUDIO_DIR)
    files = [p for p in audio_path_obj.rglob("*.wav") if p.is_file()]
    
    if not files:
        print(f"在 {AUDIO_DIR} 中没有找到任何 .wav 文件！")
        return

    print(f"开始处理 {len(files)} 个音频文件...")

    for path_obj in tqdm(files):
        path_str = str(path_obj)
        try:
            # 加载并提取
            audio = load_audio(path_str)
            emb = model.get_audio_embedding(audio) # 形状 (1, D)

            # 关键：确保是一维并存入
            v = emb[0].astype("float32")
            vectors.append(v)
            metas.append({"path": path_str})

        except Exception as e:
            print(f"\n[跳过] 处理 {path_str} 时出错: {e}")

    # 汇总数据
    if len(vectors) == 0:
        print("没有提取到任何有效向量。")
        return

    # 转换为 numpy 矩阵 (N, dim)
    vectors_np = np.stack(vectors)
    
    # 强制进行 L2 归一化（使 IndexFlatIP 等同于余弦相似度）
    faiss.normalize_L2(vectors_np)

    print(f"构建索引中，向量形状: {vectors_np.shape}")

    # 创建并保存索引
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    index = FaissIndex(dim=vectors_np.shape[1])
    index.add(vectors_np, metas)
    index.save(SAVE_PATH)

    print(f"🎉 任务完成！索引已保存至: {SAVE_PATH}.index")



In [ ]:
import faiss
import numpy as np
from collections import defaultdict



INDEX_PATH = "data/embeddings/audio_index"
MODEL_PATH = "/root/autodl-tmp/clap-model"


# ==========================================
# 1. Query扩展（Multi-Query）
# ==========================================
def expand_query(query):
    query = query.lower()
    expansions = [query]

    rules = {
        "ocean": ["sea", "wave", "beach"],
        "sea": ["ocean", "wave"],
        "rain": ["rainfall", "water drops"],
        "dog": ["barking", "puppy"],
        "fire": ["flame", "burning"],
        "wind": ["air", "storm"],
    }

    for k, vs in rules.items():
        if k in query:
            expansions.extend(vs)

    return list(set(expansions))


# ==========================================
# 2. 多查询检索
# ==========================================
def multi_search(query, model, index, topk=5):
    queries = expand_query(query)

    all_results = []

    for q in queries:
        emb = model.get_text_embedding(q)

        # 归一化（必须）
        faiss.normalize_L2(emb)

        D, I = index.index.search(emb.astype("float32"), topk)

        for idx, score in zip(I[0], D[0]):
            all_results.append((idx, float(score)))

    return all_results


# ==========================================
# 3. 结果融合（关键）
# ==========================================
def merge_results(results):
    score_dict = defaultdict(float)

    for idx, score in results:
        score_dict[idx] += score  # 累加

    ranked = sorted(score_dict.items(), key=lambda x: x[1], reverse=True)
    return ranked


# ==========================================
# 4. 简单 Rerank
# ==========================================
def rerank(ranked, metadata, query, topk=5):
    final = []

    query_words = query.lower().split()

    for idx, score in ranked[:50]:  # 先取前50再rerank
        meta = metadata[idx]
        path = meta["path"].lower()

        bonus = 0.0

        # 规则1：路径匹配关键词
        for w in query_words:
            if w in path:
                bonus += 0.05

        final.append({
            "path": meta["path"],
            "score": score + bonus
        })

    final = sorted(final, key=lambda x: x["score"], reverse=True)

    return final[:topk]


# ==========================================
# 5. 最终接口
# ==========================================
def search(query, topk=5):
    model = CLAPWrapper(MODEL_PATH)

    index = FaissIndex(dim=512)
    index.load(INDEX_PATH)

    # multi-query 检索
    raw_results = multi_search(query, model, index, topk=topk)

    # 融合
    merged = merge_results(raw_results)

    # rerank
    final = rerank(merged, index.metadata, query, topk=topk)

    return final


# ==========================================
# 6. CLI测试
# ==========================================
if __name__ == "__main__":
    while True:
        q = input("\nQuery: ")
        res = search(q)

        for r in res:
            print(f"{r['score']:.4f}  {r['path']}")

In [ ]:
import faiss
import numpy as np
import torch
import json
import os
from collections import defaultdict
from functools import lru_cache
from openai import OpenAI  # DeepSeek 兼容 OpenAI SDK
from sentence_transformers import CrossEncoder

# ==========================================
# 0. 配置与初始化 (建议单例化)
# ==========================================
DEEPSEEK_API_KEY = "your_api_key"
MODEL_PATH = "/root/autodl-tmp/clap-model"
# 推荐使用轻量级精排模型，如 BGE-Reranker-Base
RERANK_MODEL_NAME = "BAAI/bge-reranker-base" 

client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
# 本地加载 Reranker
rerank_model = CrossEncoder(RERANK_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. 语义扩展 (DeepSeek API + Cache)
# ==========================================
@lru_cache(maxsize=128)
def expand_query_llm(query):
    """
    使用 DeepSeek 将原始 Query 扩展为多个同义/近义词，增加召回覆盖率。
    lru_cache 解决 API 慢和重复调用费用问题。
    """
    prompt = f"Please provide 3-4 English synonyms or related audio descriptions for the following search term: '{query}'. Output only the terms separated by commas."
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
            temperature=0.3
        )
        content = response.choices[0].message.content
        expansions = [q.strip() for q in content.split(",") if q.strip()]
        return list(set([query] + expansions))
    except Exception as e:
        print(f"API Error: {e}, falling back to original query.")
        return [query]

# ==========================================
# 2. 结果融合 (RRF 算法)
# ==========================================
def rrf_merge(multi_results, k=60):
    """
    Reciprocal Rank Fusion: 核心在于排名而非原始分值。
    multi_results: List of List, 每一项是单次检索的 (idx, score) 列表
    """
    rrf_score = defaultdict(float)
    for results in multi_results:
        for rank, (idx, _) in enumerate(results):
            # rank 从 0 开始，公式：1 / (k + rank + 1)
            rrf_score[idx] += 1.0 / (k + rank + 1)
    
    # 按照 RRF 得分排序
    merged = sorted(rrf_score.items(), key=lambda x: x[1], reverse=True)
    return merged

# ==========================================
# 3. 深度重排 (Cross-Encoder)
# ==========================================
def deep_rerank(merged_candidates, index_metadata, query, topk=5):
    """
    使用 Cross-Encoder 对 Top-N 候选进行精排。
    """
    if not merged_candidates:
        return []

    # 取 RRF 前 20-30 个进行精排，兼顾速度与精度
    candidates = merged_candidates[:30]
    idx_list = [c[0] for c in candidates]
    
    # 准备 Cross-Encoder 输入: (Query, Passage/Metadata) 对
    # 假设你的 metadata 存了描述信息，如果没有，可以用 path 代替
    input_pairs = []
    for idx in idx_list:
        meta = index_metadata[idx]
        desc = meta.get("description", meta["path"]) # 优先用描述
        input_pairs.append([query, desc])

    # 模型打分 (推理)
    scores = rerank_model.predict(input_pairs)
    
    final_results = []
    for i, score in enumerate(scores):
        final_results.append({
            "path": index_metadata[idx_list[i]]["path"],
            "score": float(score),
            "idx": idx_list[i]
        })
    
    return sorted(final_results, key=lambda x: x["score"], reverse=True)[:topk]

# ==========================================
# 4. 完整检索链路
# ==========================================
def search_pro(query, model, index, topk=5):
    # 1. DeepSeek 扩展
    queries = expand_query_llm(query)
    
    # 2. 多路并发检索 (此处可用线程池加速)
    all_search_results = []
    for q in queries:
        emb = model.get_text_embedding(q)
        faiss.normalize_L2(emb)
        D, I = index.index.search(emb.astype("float32"), 50) # 召回多一点给精排
        all_search_results.append(list(zip(I[0], D[0])))
    
    # 3. RRF 融合
    merged = rrf_merge(all_search_results)
    
    # 4. Cross-Encoder 精排
    final = deep_rerank(merged, index.metadata, query, topk=topk)
    
    return final